- Model Used from: https://docs.pytorch.org/vision/0.28/models/convnext.html
- Convnext Architecture: https://www.geeksforgeeks.org/computer-vision/convnext/

### Findings:
With my RTX 5070 TI I am able to train with:
- Batch Size = 128
- Image Sizes = 256 x 256
- Pytorch ConvNeXt Tiny Model

In [1]:
import time
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import models, datasets, transforms


In [ ]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 256
TRAIN_SAMPLES = None # use a smaller training subset for a quick GPU test
DATALOADER_WORKERS = 4 # 4 because my CPU has 8 cores available

generator = torch.Generator().manual_seed(42)

# select device based on machine
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
# need to upscale images, and create 3 channel RGB images for convnext model
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # upscale image
    transforms.Grayscale(num_output_channels=3), # convert 1 channel to 3 channels
    transforms.ToTensor(),
])

# download MNIST train dataset and transform it
full_train = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
# download MNIST test dataset and transform it
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# split training set into 80% train, and 20% validation
train_dataset, val_dataset = random_split(full_train, [0.8, 0.2], generator=generator)

# use only part of the training set for a quick GPU backpropagation test
if TRAIN_SAMPLES is not None:
    train_dataset = torch.utils.data.Subset(
        train_dataset,
        range(min(TRAIN_SAMPLES, len(train_dataset))),
    )

# dataloaders for each dataset, train dataset shuffles each epoch
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=DATALOADER_WORKERS, # number of CPU workers for preparing batches
    pin_memory=True, # allow for fast data transfer to GPU
    persistent_workers=True, # prevent CPU from restarting workers
    generator=torch.Generator().manual_seed(42),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=DATALOADER_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=DATALOADER_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)


print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1024
Validation: 12000
Test: 10000


In [4]:
# initalise ConvNeXt model with random weights
model = models.convnext_tiny(weights=None)

# get the input features of the final layer
num_features = model.classifier[2].in_features

# replace the final layer with only 10 output classes
num_classes = 10
model.classifier[2] = nn.Linear(num_features, num_classes)

# move to GPU
model = model.to(device)

# define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [ ]:
# training loop
for epoch in range(EPOCHS):
    epoch_start_time = time.perf_counter()
    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0
    total_train_batches = len(train_loader)

    for batch_index, (inputs, labels) in enumerate(train_loader, start=1):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # zero the parameter gradients
        outputs = model(inputs) # forward pass through network
        loss = criterion(outputs, labels) # calculate loss
        loss.backward() # compute gradients
        optimizer.step() # update weights

        running_loss += loss.item() * inputs.size(0)
        correct_train += (outputs.argmax(dim=1) == labels).sum().item()
        total_train += labels.size(0)

        print(
            f"Epoch {epoch+1}/{EPOCHS} - "
            f"Step {batch_index}/{total_train_batches} "
            f"({batch_index / total_train_batches:.0%}) - "
            f"Loss: {loss.item():.4f}",
            end="\r",
            flush=True,
        )

    train_loss = running_loss / len(train_dataset)
    train_accuracy = correct_train / total_train

    # run validation on current epoch
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad(): # no need to track gradient computations for validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs) # forward pass
            loss = criterion(outputs, labels) # calculate validation loss

            running_val_loss += loss.item() * inputs.size(0)
            correct_val += (outputs.argmax(dim=1) == labels).sum().item()
            total_val += labels.size(0)

    val_loss = running_val_loss / len(val_dataset)
    val_accuracy = correct_val / total_val
    epoch_time = time.perf_counter() - epoch_start_time

    # print current epoch losses and duration
    print(
        f"Epoch {epoch+1}/{EPOCHS} complete - "
        f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f} - "
        f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f} - "
        f"Time: {epoch_time:.2f}s"
    )

Epoch 1/10 complete - Train Loss: 2.5214, Train Accuracy: 0.1650 - Validation Loss: 2.4838, Validation Accuracy: 0.1093 - Time: 109.94s
